In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/raw/LI-Small_Trans.csv",
    nrows=100000
)

df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,11,8000ECA90,11,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,3402,80021DAD0,3402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,11,8000ECA90,1120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,3814,8006AD080,3814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,20,8006AD530,20,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [3]:
duplicates = df.duplicated().sum()

print("Duplicate rows :", duplicates)

Duplicate rows : 0


In [4]:
df = df.drop_duplicates()

print(df.shape)

(100000, 11)


In [5]:
df.isnull().sum()

Timestamp             0
From Bank             0
Account               0
To Bank               0
Account.1             0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64

In [6]:
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

In [7]:
df["Hour"] = df["Timestamp"].dt.hour

In [8]:
df["Day"] = df["Timestamp"].dt.day_name()

In [9]:
df["Month"] = df["Timestamp"].dt.month

In [10]:
df["Weekend"] = df["Day"].isin(["Saturday","Sunday"])

In [11]:
df["Amount Difference"] = (
    df["Amount Received"] -
    df["Amount Paid"]
)

In [12]:
df["Currency Match"] = (
    df["Receiving Currency"] ==
    df["Payment Currency"]
)

In [13]:
df["Same Bank"] = (
    df["From Bank"] ==
    df["To Bank"]
)

In [14]:
df = df.sort_values("Timestamp")

df["Time Difference"] = (
    df["Timestamp"]
    .diff()
    .dt.total_seconds()
)

In [15]:
threshold = df["Amount Paid"].quantile(0.99)

df["Large Transaction"] = (
    df["Amount Paid"] >= threshold
)

In [16]:
sender_stats = (
    df.groupby("Account")
      .agg(
          Total_Sent=("Amount Paid","sum"),
          Avg_Sent=("Amount Paid","mean"),
          Transaction_Count=("Amount Paid","count")
      )
)

sender_stats.head()

,Total_Sent,Avg_Sent,Transaction_Count
Account,,,
10042B660,3.420183e+09,1.806753e+06,1893
800043BE0,1.615711e+04,1.615711e+04,1
800043DA0,3.528196e+04,7.056392e+03,5
800044230,4.375443e+04,4.375443e+04,1
800044900,1.335959e+06,4.453195e+05,3


In [17]:
receiver_stats = (
    df.groupby("Account.1")
      .agg(
          Total_Received=("Amount Received","sum"),
          Avg_Received=("Amount Received","mean")
      )
)

receiver_stats.head()

,Total_Received,Avg_Received
Account.1,,
10042B660,0.03,0.015
800044230,43754.43,43754.430
800054200,700620.29,700620.290
800054290,891489.28,891489.280
800056090,7666.72,7666.720


In [18]:
sender_stats = sender_stats.reset_index()

receiver_stats = receiver_stats.reset_index()

account_summary = sender_stats.merge(
    receiver_stats,
    left_on="Account",
    right_on="Account.1",
    how="outer"
)

account_summary.head()

,Account,Total_Sent,Avg_Sent,Transaction_Count,Account.1,Total_Received,Avg_Received
0,10042B660,3.420183e+09,1.806753e+06,1893.0,10042B660,0.03,0.015
1,800043BE0,1.615711e+04,1.615711e+04,1.0,NaN,NaN,NaN
2,800043DA0,3.528196e+04,7.056392e+03,5.0,NaN,NaN,NaN
3,800044230,4.375443e+04,4.375443e+04,1.0,800044230,43754.43,43754.430
4,800044900,1.335959e+06,4.453195e+05,3.0,NaN,NaN,NaN


In [19]:
df.to_csv(
    "../data/processed/transactions_processed.csv",
    index=False
)

In [20]:
account_summary.to_csv(
    "../data/processed/account_summary.csv",
    index=False
)